In [36]:
import csv , pandas as pd 
import psycopg2
from psycopg2.extras import execute_values
import numpy as np
import io
from datetime import datetime
from uuid import UUID
import re
import random

In [37]:
file_path = 'event_log_master_sap_1.csv'
bad_lines = []

with open(file_path, mode='r', encoding='utf-16') as f:
    reader = csv.reader(f, delimiter=',')
    header = next(reader)
    expected_fields = len(header)  # 46 columns

    for line_num, row in enumerate(reader, start=2): # start=2 (1-indexed after header)
        if len(row) != expected_fields:
            bad_lines.append((line_num, len(row), row))

print(f"Total bad lines found: {len(bad_lines)}\n")
for line_num, field_count, row in bad_lines:
    print(f"Line {line_num}: Found {field_count} fields (expected {expected_fields})")
    print(f"Row data: {row}\n")


Total bad lines found: 0



In [38]:
kmat = pd.read_csv('KMAT.csv',delimiter="\t")
kmat

,Old material no.,Material description,Language,New Material
0,001A,LT-25 - FANUC,EN,40001
1,001B,LT-25 - SIEMENS,EN,40002
2,002A,LT-25 630 - FANUC,EN,40003
3,003A,LT-20 C CLASSIC - FANUC,EN,40004
4,003B,LT-20 C CLASSIC - SIEMENS,EN,40005
...,...,...,...,...
566,SPARK,SPARK,EN,40514
567,SPARKXL,SPARKXL,EN,40515
568,SUPERWINNER,SUPERWINNER,EN,40516
569,TCV-540,TCV540,EN,40517


In [39]:
# postgres connection
conn = None
import psycopg2

conn = None
try:
    conn = psycopg2.connect(
        dbname="miot_cnc_dev2",
        user="postgres",
        password="password",
        host="172.36.0.107",
        port="5433"
    )
    print("Database connected successfully!")
except psycopg2.Error as e:
    print(f"Database connection error: {e}")


Database connected successfully!


In [40]:
import pandas as pd

df = pd.read_csv(
    'event_log_master_sap_28072026.csv',
    delimiter=",",
    encoding='utf-16'
)


In [41]:
# 1. segregate out Cancelled Machines
cancelled_df = df[df['CancelIndicator'] == 'X']
cancelled_df.to_csv('cancelled_machines.csv',index=False,quoting=csv.QUOTE_ALL)

df = df[df['CancelIndicator'].isna()].reset_index(drop=True)

In [42]:
# 2. find duplicates of cnc id machines
duplicates_cnc = df[df.duplicated(subset=['CNCID'],keep=False) & df['CNCID'].notna()]
duplicates_cnc.to_csv('duplicated_cnc.csv',index=False,quoting=csv.QUOTE_ALL)

null_cnc = df[df['CNCID'].isnull()]
null_cnc.to_csv('null_cnc.csv',index=False,quoting=csv.QUOTE_ALL)


df = df[df['CNCID'].notna() & ~df.duplicated(subset=['CNCID'],keep=False)].reset_index(drop=False)
df

,index,IDD,Branch,Location,Customer,Supplier,SupplierProduct,CustomerProductSlno,DispatchDate,CommissionedDate,...,ShipToName,ShipToAddress,ShipToCity,ShipToPincode,CreditAvailed,CreditStartDate,CreditEndDate,AlertPhoneNumbers,AlertMailIDs,Region
0,0,1897,BOMMASANDRA,HOSUR,Mega Milies Bearing cups Pvt Ltd,TCD,JOBBER JUNIOR,40048-10919,2024-08-03 00:00:00,2024-08-05 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Karnataka
1,1,1898,AMBATTUR,AMBATTUR,Globe Components Pvt Ltd,TCD,CUB L M,40085-1803,2024-08-19 00:00:00,2024-08-20 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TamilNadu
2,2,1899,AMBATTUR,AMBATTUR,Globe Components Pvt Ltd,TCD,CUB LM,40085-1802,2024-08-19 00:00:00,2024-08-20 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TamilNadu
3,3,1900,BOMMASANDRA,HOSUR,Murugan Industries,TCD,J 300 LM,40209-11158,2024-08-20 00:00:00,2024-08-21 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Karnataka
4,4,1901,BOMMASANDRA,HOSUR,S G A Industries,TCD,J 300 LM,40209-11143,2024-08-21 00:00:00,2024-08-22 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Karnataka
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13797,14293,26385,Ramanathapuram,Ramanathapuram,Perfect Engineering Products,MCD,MCV450 PLUS,40553-1057,2026-07-27 00:00:00,NaN,...,Perfect Engineering Products,"123,124, Industrial Estate,,SIDCO,,Malumachamp...",Coimbatore,641050,No,NaN,NaN,NaN,NaN,NaN
13798,14294,26386,Rohtak,Rohtak,Shri Krishna Forgings,TCD,SJE 06 RG - FANUC,40281-6745,2026-07-27 00:00:00,NaN,...,Shri Krishna Forgings,"11 KM Stone,,Rohtak-Bhiwani Road,,Banyani,",Rohtak,124001,No,NaN,NaN,NaN,NaN,NaN
13799,14295,26387,Rohtak,Rohtak,Shri Krishna Forgings,TCD,SJE 06 RG - FANUC,40281-6746,2026-07-27 00:00:00,NaN,...,Shri Krishna Forgings,"11 KM Stone,,Rohtak-Bhiwani Road,,Banyani,",Rohtak,124001,No,NaN,NaN,NaN,NaN,NaN
13800,14297,26389,Wagholi,Wagholi,Alto Industry,TCD,J 300 LM - FANUC,40209-14033,2026-07-27 00:00:00,NaN,...,Alto Industry,"Gat No.577,,Opposite Seco Tools,,Pune Nagar Ro...",Pune,412216,No,NaN,NaN,NaN,NaN,NaN


In [43]:
# Keep only rows where CNCID has 4 groups of exactly 8 characters
pattern = r'^[^-]{8}(?:-[^-]{8}){3}$'

df = df[
    df["CNCID"].astype(str).str.match(pattern, na=False)
].copy()

In [44]:
# 3. Update Supplier 'ACE' -> 'MCD' and  'AMS' -> 'TCD'
df['Supplier'] = df['Supplier'].replace({
    'ACE':'MCD',
    'AMS':'TCD'
})

In [45]:
# ------------------------------------------------------------------
# 1. Clean input columns
# ------------------------------------------------------------------
df['SupplierProduct'] = df['SupplierProduct'].astype(str).str.strip()
df['CustomerProductSlno'] = df['CustomerProductSlno'].astype(str).str.strip()

# ------------------------------------------------------------------
# 2. KMAT lookup tables
# ------------------------------------------------------------------
kmat_desc_lookup = kmat[
    ['Material description', 'Old material no.', 'New Material']
].drop_duplicates(subset=['Material description'])

kmat_newmat_lookup = kmat[
    ['Material description', 'New Material', 'Old material no.']
].drop_duplicates(subset=['New Material'])

# ------------------------------------------------------------------
# 3. FIRST: Direct match on Material Description
# ------------------------------------------------------------------
direct_df = df.merge(
    kmat_desc_lookup,
    left_on='SupplierProduct',
    right_on='Material description',
    how='inner'
)

direct_df['MappingType'] = 0

# ------------------------------------------------------------------
# 4. Keep only unmatched rows
# ------------------------------------------------------------------
remaining_df = df[
    ~df['SupplierProduct'].isin(direct_df['SupplierProduct'])
].copy()

# ------------------------------------------------------------------
# 5. Existing logic starts here (only on unmatched rows)
# ------------------------------------------------------------------
kmat_descriptions = set(
    kmat['Material description'].dropna().astype(str).str.strip()
)
kmat_new_materials = set(
    kmat['New Material'].dropna().astype(str).str.strip()
)

supplier_prod = remaining_df['SupplierProduct'].astype(str).str.strip()
customer_slno = remaining_df['CustomerProductSlno'].astype(str).str.strip()

has_hyphen = customer_slno.str.contains('-', na=False)
slno_prefix = customer_slno.str.split('-').str[0].str.strip()

match_description = supplier_prod.isin(kmat_descriptions)
match_new_material = has_hyphen & slno_prefix.isin(kmat_new_materials)

# Keep only rows matching KMAT
remaining_df = remaining_df[
    match_description | match_new_material
].reset_index(drop=True)

# Prepare keys
remaining_df['SupplierProduct'] = remaining_df['SupplierProduct'].astype(str).str.strip()
remaining_df['CustomerProductSlno'] = remaining_df['CustomerProductSlno'].astype(str).str.strip()
remaining_df['Prefix'] = remaining_df['CustomerProductSlno'].str.split('-').str[0]

# Split into two groups
desc_df = remaining_df[
    ~remaining_df['CustomerProductSlno'].str.contains('-', na=False)
].copy()

newmat_df = remaining_df[
    remaining_df['CustomerProductSlno'].str.contains('-', na=False)
].copy()

# Merge by Material description
desc_df = desc_df.merge(
    kmat_desc_lookup,
    left_on='SupplierProduct',
    right_on='Material description',
    how='left',
)

# Merge by New Material
newmat_df = newmat_df.merge(
    kmat_newmat_lookup,
    left_on='Prefix',
    right_on='New Material',
    how='left'
)

# Mapping type indicator
desc_df['MappingType'] = 0
newmat_df['MappingType'] = 1

# ------------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------------
df = pd.concat(
    [direct_df, desc_df, newmat_df],
    ignore_index=True
)

df.drop(columns=['Prefix'], inplace=True, errors='ignore')

# Use official KMAT Material description
df['SupplierProduct'] = df['Material description'].fillna(df['SupplierProduct'])

print("Total records:", len(df))

Total records: 12699


In [46]:
# feature hardcdoed
df['Features'] = 10000000

In [47]:
# 7. updated Invoice and InvoiceItem with random Values
used = set(df['Invoice'].dropna().astype(str))

def random_invoice():
    while True:
        inv = str(random.randint(100, 999))
        if inv not in used:
            used.add(inv)
            return inv

invoice_updates = {}
invoiceitem_updates = {}

for (customer, cncid), grp in df.groupby(['Customer', 'CNCID'], sort=False):

    # -------- Invoice --------
    inv = None

    # Reuse existing invoice if present in this CNC group
    existing = grp['Invoice'].dropna()
    if not existing.empty:
        inv = str(existing.iloc[0])
    else:
        inv = random_invoice()

    # Fill only missing invoices
    for idx in grp[grp['Invoice'].isna()].index:
        invoice_updates[idx] = inv

    # -------- InvoiceItem --------
    item_rows = grp[grp['InvoiceItem'].isna()].index

    for i, idx in enumerate(item_rows, start=1):
        invoiceitem_updates[idx] = str(i * 10)

# Bulk update (very fast)
df.loc[invoice_updates.keys(), 'Invoice'] = list(invoice_updates.values())
df.loc[invoiceitem_updates.keys(), 'InvoiceItem'] = list(invoiceitem_updates.values())

In [48]:
df.to_csv('cleaned.csv',sep=',')

In [49]:
brz_df = df.copy()

brz_df = brz_df.rename(
    columns={
        "Branch": "branch",
        "Location": "location",
        "Customer": "customer",
        "Supplier": "supplier",
        "SupplierProduct": "supplier_product",
        "CustomerProductSlno": "customer_product_slno",
        "DispatchDate": "dispatch_date",
        "CommissionedDate": "commissioned_date",
        "CustomerProductStatus": "customer_product_status",
        "Features": "features",
        "CNCID": "cnc_id",
        "UpdatedBy": "updated_by",
        "UpdatedTS": "updated_ts",
        "Invoice": "invoice",
        "CancelIndicator": "cancel_indicator",
        "InvoiceItem": "invoice_item",
        "CustomerNo": "customer_no",
        "CustomerAddress": "customer_address",
        "CustomerCity": "customer_city",
        "CustomerPincode": "customer_pincode",
        "ShipToNo": "ship_to_no",
        "ShipToName": "ship_to_name",
        "ShipToAddress": "ship_to_address",
        "ShipToCity": "ship_to_city",
        "ShipToPincode": "ship_to_pincode",
    }
)

# Convert dates & timestamps
brz_df["dispatch_date"] = pd.to_datetime(
    brz_df["dispatch_date"], errors="coerce"
).dt.date
brz_df["commissioned_date"] = pd.to_datetime(
    brz_df["commissioned_date"], errors="coerce"
).dt.date
brz_df["updated_ts"] = pd.to_datetime(brz_df["updated_ts"], errors="coerce")

# Add DB metadata columns
brz_df["job_id"] = 13
brz_df["ingested_ts"] = pd.Timestamp.now()
brz_df["created_at"] = pd.Timestamp.now()

# Define DB column order
db_columns = [
    "job_id",
    "branch",
    "location",
    "customer",
    "supplier",
    "supplier_product",
    "customer_product_slno",
    "dispatch_date",
    "commissioned_date",
    "customer_product_status",
    "features",
    "cnc_id",
    "updated_by",
    "updated_ts",
    "invoice",
    "cancel_indicator",
    "invoice_item",
    "customer_no",
    "customer_address",
    "customer_city",
    "customer_pincode",
    "ship_to_no",
    "ship_to_name",
    "ship_to_address",
    "ship_to_city",
    "ship_to_pincode",
    "ingested_ts",
    "created_at",
]

brz_df = brz_df[db_columns]
brz_df = brz_df.replace(r"^\s*$", np.nan, regex=True)
brz_df = brz_df.where(pd.notnull(brz_df), None)

print("brz_df shape:", brz_df.shape)
print("Sample supplier_product values:")
print(brz_df[["supplier_product", "customer_product_slno", "job_id"]].head())

brz_df shape: (12699, 28)
Sample supplier_product values:
          supplier_product customer_product_slno  job_id
0                     430V             40469-336      13
1      VANTAGE 500 - FANUC                   549      13
2  LC-12 / COLT LM - FANUC                   979      13
3  LC-12 / COLT LM - FANUC                   980      13
4      VANTAGE 500 - FANUC                   550      13


In [50]:
brz_df
brz_df.to_csv('temp.csv',sep=',')

In [51]:
# buffer = io.StringIO()
# brz_df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
# buffer.seek(0)

# # 2. Define the COPY command
# copy_sql = r"""
# COPY bronze.raw_machine_dispatch_logs (
#     job_id,
#     branch,
#     location,
#     customer,
#     supplier,
#     supplier_product,
#     customer_product_slno,
#     dispatch_date,
#     commissioned_date,
#     customer_product_status,
#     features,
#     cnc_id,
#     updated_by,
#     updated_ts,
#     invoice,
#     cancel_indicator,
#     invoice_item,
#     customer_no,
#     customer_address,
#     customer_city,
#     customer_pincode,
#     ship_to_no,
#     ship_to_name,
#     ship_to_address,
#     ship_to_city,
#     ship_to_pincode,
#     ingested_ts,
#     created_at
# ) FROM STDIN WITH (FORMAT csv, DELIMITER E'\t', NULL '\N');
# """

# # 3. Execute Bulk COPY
# with conn.cursor() as cur:
#     cur.copy_expert(sql=copy_sql, file=buffer)

# conn.commit()

# print("Bulk COPY inserted successfully!")


In [52]:
now = datetime.now()
zero_uuid = "00000000-0000-0000-0000-000000000000"

# 1. Extract columns and rename
upsert_df = (
    df[
        [
            "Supplier",
            "Material description",
            "Old material no.",
            "New Material",
        ]
    ]
    .dropna(subset=["Material description"])
    .copy()
    .rename(
        columns={
            "Material description": "supplier_product",
            "Old material no.": "old_material_no",
            "New Material": "new_material_no",
        }
    )
)

# 2. Supplier mapping (MCD -> 1, TCD -> 2)
upsert_df["supplier_id"] = upsert_df["Supplier"].apply(
    lambda x: 1 if str(x).strip().upper() == "MCD" else 2
)

# 3. Deduplicate on (supplier_id, supplier_product) to match SQL ON CONFLICT target
upsert_df = upsert_df.drop_duplicates(
    subset=["supplier_id", "supplier_product"]
)

# 4. Clean strings and handle nulls
for col in ["supplier_product", "old_material_no", "new_material_no"]:
  upsert_df[col] = (
      upsert_df[col]
      .astype(str)
      .str.strip()
      .replace({"nan": None, "None": None, "": None})
  )

# 5. Audit columns
upsert_df["created_by"] = zero_uuid
upsert_df["updated_by"] = zero_uuid
upsert_df["created_at"] = now
upsert_df["updated_at"] = now
upsert_df["updated_type"] = "service"

# 6. Final column order
upsert_df = upsert_df[
    [
        "supplier_id",
        "supplier_product",
        "created_by",
        "updated_by",
        "created_at",
        "updated_at",
        "updated_type",
        "old_material_no",
        "new_material_no",
    ]
]

# 7. Convert remaining NaNs -> None
upsert_df = upsert_df.where(upsert_df.notna(), None)

# 8. Convert to tuple list for SQL insertion
records = list(upsert_df.itertuples(index=False, name=None))

In [53]:
# sql = """
# INSERT INTO sap.dim_product (
#     supplier_id,
#     supplier_product,
#     created_by,
#     updated_by,
#     created_at,
#     updated_at,
#     updated_type,
#     old_material_no,
#     new_material_no
# )
# VALUES %s
# ON CONFLICT (supplier_id,supplier_product)
# DO UPDATE SET
#     supplier_id     = EXCLUDED.supplier_id,
#     old_material_no = EXCLUDED.old_material_no,
#     new_material_no = EXCLUDED.new_material_no,
#     updated_by      = EXCLUDED.updated_by,
#     updated_at      = EXCLUDED.updated_at,
#     updated_type    = EXCLUDED.updated_type;
# """

# with conn.cursor() as cur:
#     execute_values(cur, sql, records, page_size=1000)

# conn.commit()

In [54]:
# df['DispatchDate'].isna()
temp = df[~df['Invoice'].isna() & df['InvoiceItem'].isna()]
temp[['CNCID','Invoice','InvoiceItem']]

# Get count of CNCID per Invoice directly
invoice_cnc_counts = temp['Invoice'].value_counts().reset_index()
invoice_cnc_counts.columns = ['Invoice', 'cnc_count']

print(invoice_cnc_counts.head())


Empty DataFrame
Columns: [Invoice, cnc_count]
Index: []


In [55]:
# Fetch existing Branch/Location combinations
with conn.cursor() as cur:
    cur.execute("""
        SELECT branch_name, location_name
        FROM sap.dim_branch
    """)
    db_combos = {
        (branch.strip(), location.strip())
        for branch, location in cur.fetchall()
    }

# Get unique combinations from Excel
branch_df = (
    df[['Branch', 'Location', 'Region']]
    .dropna(subset=['Branch', 'Location'])
    .copy()
)

# Strip whitespace
branch_df["Branch"] = branch_df["Branch"].astype(str).str.strip()
branch_df["Location"] = branch_df["Location"].astype(str).str.strip()
branch_df["Region"] = branch_df["Region"].astype(str).str.strip()

# Remove duplicates after stripping
branch_df = branch_df.drop_duplicates(subset=["Branch", "Location"])

# Keep only rows not present in DB
missing_df = branch_df[
    ~pd.MultiIndex.from_frame(
        branch_df[["Branch", "Location"]]
    ).isin(db_combos)
].copy()

print(f"Rows to insert: {len(missing_df)}")

# Add audit columns
missing_df["created_by"] = zero_uuid
missing_df["updated_by"] = zero_uuid
missing_df["created_at"] = now
missing_df["updated_at"] = now
missing_df["updated_type"] = "service"

# Create records
records = list(
    missing_df[
        [
            "Branch",
            "Location",
            "created_by",
            "updated_by",
            "created_at",
            "updated_at",
            "Region",
            "updated_type",
        ]
    ].itertuples(index=False, name=None)
)

Rows to insert: 0


In [56]:
# sql = """
# INSERT INTO sap.dim_branch
# (
#     branch_name,
#     location_name,
#     created_by,
#     updated_by,
#     created_at,
#     updated_at,
#     region,
#     updated_type
# )
# VALUES %s
# ON CONFLICT (branch_name, location_name) DO NOTHING;
# """

# with conn.cursor() as cur:
#     execute_values(cur, sql, records, page_size=1000)

# conn.commit()

# print(f"Inserted {len(records)} new branch/location combinations.")

In [65]:
# Keep only rows with ActivationKey
activation_df = df[df["ActivationKey"].notna()].copy()

# Validate ActivationKey format: XXXX-XXXX-XXXX-XXXX
pattern = re.compile(r'^[A-Za-z0-9]{4}(?:-[A-Za-z0-9]{4}){3}$')

activation_df = activation_df[
    activation_df["ActivationKey"].astype(str).str.match(pattern)
].copy()

# Activate machine
activation_df["IsActivated"] = 1

# Default License Type
activation_df["LicenseType"] = "Standard"

# Default Customer Plant ID
activation_df["CustPlantID"] = "PLANT-01"

# Convert DispatchDate to datetime
activation_df["DispatchDate"] = pd.to_datetime(
    activation_df["DispatchDate"], errors="coerce"
)

# Fill YearOfManufacturing from DispatchDate if missing or 0
mask = (
    activation_df["YearOfManufacturing"].isna()
    | (activation_df["YearOfManufacturing"] == 0)
)

activation_df.loc[mask, "YearOfManufacturing"] = (
    activation_df.loc[mask, "DispatchDate"].dt.year.astype("Int64")
)

# Keep only required columns
activation_df = activation_df[
    [
        "CNCID",
        "ActivationKey",
        "IsActivated",
        "LicenseType",
        "SupportNumber",
        "YearOfManufacturing",
        "WarrantyExpireDate",
        "CustAssetNumber",
        "CustPlantID",
        "ActivationAck",
        "AwarenessTrainingGivenOrNot",
    ]
]

activation_df

,CNCID,ActivationKey,IsActivated,LicenseType,SupportNumber,YearOfManufacturing,WarrantyExpireDate,CustAssetNumber,CustPlantID,ActivationAck,AwarenessTrainingGivenOrNot
0,3C7B5D01-4CA192AB-BE322789-6965C4FE,E394-2758-0549-90EA,1,Standard,NaN,2024.0,2025-09-05 00:00:00,336,PLANT-01,NaN,NaN
1,3C7B5D01-EDA172CB-BE312789-6965C4DF,E39E-1758-0547-90FC,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN
2,3C7B5D01-49A18DAC-BE312789-6B65C4E2,E3B4-1758-0548-902A,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN
3,3C7B5D01-D6A127C5-BE312789-6B65C4BE,E3BD-1758-0542-90EC,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN
4,3C7B5D01-44A157B3-BE312789-6965C428,E394-1758-0545-908B,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
12694,3C7B5D01-2DA19508-BE312789-6865C439,E382-1758-0549-9090,1,Standard,91,2025.0,2026-06-22 00:00:00,1,PLANT-01,True,True
12695,3C7B5D01-3DA1D518-BE322789-6665C474,E363-2758-054D-9041,1,Standard,9503040941,2026.0,2027-05-14 00:00:00,2,PLANT-01,NaN,NaN
12696,3C7B5D01-00A11077-BE312789-6665C4E0,E360-1758-0541-9007,1,Standard,9840785520,2026.0,2027-07-19 00:00:00,7777,PLANT-01,True,True
12697,3C7B5D01-02A17A69-BE312789-6665C496,E360-1758-0547-9066,1,Standard,9840785523,2026.0,2026-07-22 00:00:00,Nil,PLANT-01,NaN,True


In [66]:
# Fetch CNCID, Dispatch ID and Customer Name from DB
query = """
SELECT
    fmdl.cnc_id,
    fmdl.dispatch_id,
    dc.customer_name
FROM sap.fact_machine_dispatch_logs fmdl
INNER JOIN sap.dim_customer dc
    ON fmdl.customer_id = dc.customer_id
WHERE fmdl.job_id = 15;
"""

with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

customer_df = pd.DataFrame(rows, columns=columns)

# Rename for merge
customer_df.rename(columns={"cnc_id": "CNCID"}, inplace=True)

# Remove duplicates
customer_df = customer_df.drop_duplicates(subset=["CNCID"])

# Remove existing columns if present
activation_df = activation_df.drop(
    columns=["customer_name", "dispatch_id"],
    errors="ignore"
)

# Merge
activation_df = activation_df.merge(
    customer_df,
    on="CNCID",
    how="left"
)

activation_df["ActivatedByEmailID"] = "supportteam@gmail.com"

activation_df

,CNCID,ActivationKey,IsActivated,LicenseType,SupportNumber,YearOfManufacturing,WarrantyExpireDate,CustAssetNumber,CustPlantID,ActivationAck,AwarenessTrainingGivenOrNot,dispatch_id,customer_name,ActivatedByEmailID
0,3C7B5D01-4CA192AB-BE322789-6965C4FE,E394-2758-0549-90EA,1,Standard,NaN,2024.0,2025-09-05 00:00:00,336,PLANT-01,NaN,NaN,63154,SENSARA PLANT -02,supportteam@gmail.com
1,3C7B5D01-EDA172CB-BE312789-6965C4DF,E39E-1758-0547-90FC,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN,70675,RELIABLE AUTOTECH PVT. LTD,supportteam@gmail.com
2,3C7B5D01-49A18DAC-BE312789-6B65C4E2,E3B4-1758-0548-902A,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN,62961,SHREE PRESSING,supportteam@gmail.com
3,3C7B5D01-D6A127C5-BE312789-6B65C4BE,E3BD-1758-0542-90EC,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN,69635,SHREE PRESSING,supportteam@gmail.com
4,3C7B5D01-44A157B3-BE312789-6965C428,E394-1758-0545-908B,1,Standard,NaN,2024.0,1900-01-01 00:00:00,NaN,PLANT-01,NaN,NaN,62835,RELIABLE AUTOTECH PVT. LTD,supportteam@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10641,3C7B5D01-2DA19508-BE312789-6865C439,E382-1758-0549-9090,1,Standard,91,2025.0,2026-06-22 00:00:00,1,PLANT-01,True,True,61884,SWARA ADVANCED ENGINEERING PRODUCTS,supportteam@gmail.com
10642,3C7B5D01-3DA1D518-BE322789-6665C474,E363-2758-054D-9041,1,Standard,9503040941,2026.0,2027-05-14 00:00:00,2,PLANT-01,NaN,NaN,62484,WARDHAN INDUSTRIES,supportteam@gmail.com
10643,3C7B5D01-00A11077-BE312789-6665C4E0,E360-1758-0541-9007,1,Standard,9840785520,2026.0,2027-07-19 00:00:00,7777,PLANT-01,True,True,58717,ACE,supportteam@gmail.com
10644,3C7B5D01-02A17A69-BE312789-6665C496,E360-1758-0547-9066,1,Standard,9840785523,2026.0,2026-07-22 00:00:00,Nil,PLANT-01,NaN,True,84056,GLOBAL AUTO COMPONENTS,supportteam@gmail.com


In [67]:
results = []

sql = """
SELECT *
FROM miot_support.activate_cnc_id(
    %s,  -- p_cnc_id
    %s,  -- p_activation_key
    %s,  -- p_is_activated
    %s,  -- p_license_type
    %s,  -- p_support_number
    %s,  -- p_year_of_manufacturing
    %s,  -- p_warranty_date
    %s,  -- p_customer_assest_number
    %s,  -- p_customer_plant_id
    %s,  -- p_activated_by_email_id
    %s   -- p_customer_name
);
"""

with conn.cursor() as cur:

    for _, row in activation_df.iterrows():

        cur.execute(
            sql,
            (
                str(row["CNCID"]),
                str(row["ActivationKey"]),
                bool(row["IsActivated"]),
                str(row["LicenseType"]),
                None if pd.isna(row["SupportNumber"]) else str(row["SupportNumber"]),
                None if pd.isna(row["YearOfManufacturing"]) else int(row["YearOfManufacturing"]),
                None if pd.isna(row["WarrantyExpireDate"]) else pd.to_datetime(row["WarrantyExpireDate"]).date(),
                None if pd.isna(row["CustAssetNumber"]) else str(row["CustAssetNumber"]),
                str(row["CustPlantID"]),
                str(row["ActivatedByEmailID"]),
                None if pd.isna(row["customer_name"]) else str(row["customer_name"]),
            ),
        )

        v_msg, v_is_error = cur.fetchone()

        results.append(
            {
                "dispatch_id": row["dispatch_id"],
                "v_msg": v_msg,
                "v_is_error": v_is_error,
            }
        )

conn.commit()

# Create result DataFrame
result_df = pd.DataFrame(results)

result_df

,dispatch_id,v_msg,v_is_error
0,63154,CNC ID 3C7B5D01-4CA192AB-BE322789-6965C4FE act...,False
1,70675,CNC ID 3C7B5D01-EDA172CB-BE312789-6965C4DF act...,False
2,62961,CNC ID 3C7B5D01-49A18DAC-BE312789-6B65C4E2 act...,False
3,69635,CNC ID 3C7B5D01-D6A127C5-BE312789-6B65C4BE act...,False
4,62835,CNC ID 3C7B5D01-44A157B3-BE312789-6965C428 act...,False
...,...,...,...
10641,61884,CNC ID 3C7B5D01-2DA19508-BE312789-6865C439 act...,False
10642,62484,CNC ID 3C7B5D01-3DA1D518-BE322789-6665C474 act...,False
10643,58717,CNC ID 3C7B5D01-00A11077-BE312789-6665C4E0 act...,False
10644,84056,CNC ID 3C7B5D01-02A17A69-BE312789-6665C496 act...,False


In [70]:
result_df[result_df['v_is_error'] == True]

,dispatch_id,v_msg,v_is_error
